# FFA Seasonality Charts

This script allows you to plot seasonality charts for a specific contract month from our Spark30FFA and Spark25FFA freight rates.

For a full explanation of how to import our FFA data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html


### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/lng-freight/contracts.html

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already. Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/request/authentication.html


The code then prints the available prices that are callable from the API, and their corresponding Python ticker names are displayed as a list at the bottom of the Output.

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Defining functions to fetch data

In [ ]:
# Function for collecting the list of contracts
def list_contracts(access_token):
    """
    Fetch available contracts. Return contract ticker symbols

    # Procedure:

    Do a GET query to /v1.0/contracts/ with a Bearer token authorization HTTP header.
    """
    content = do_api_get_query(uri="/v1.0/contracts/", access_token=access_token)

    print(">>>> All the contracts you can fetch")
    tickers = []
    for contract in content["data"]:
        print(contract["fullName"])
        tickers.append(contract["id"])

    return tickers

# Function to call historical data
def fetch_historical_price_releases(access_token, ticker, limit, offset=None):

    print(">>>> Get price releases for {}".format(ticker))

    query_params = "?limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)

    content = do_api_get_query(
        uri="/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params),
        access_token=access_token,
    )

    data = content["data"]

    return data

# Calling ref data function:
tickers = list_contracts(access_token)


### Function to call data and store as a DataFrame

In [ ]:
# Defining the function
def fetch_ffa_prices(my_tick, my_lim):
    print(my_tick)

    hist_data = fetch_historical_price_releases(access_token, my_tick, limit=my_lim)

    stored_data = {
        "ReleaseDate": [],
        "Ticker": [],
        "PeriodStart": [],
        "PeriodEnd": [],
        "USDperday": [],
        "USDperdayMax": [],
        "USDperdayMin": [],
    }

    for release in hist_data:

        data_points = release["data"][0]["dataPoints"]

        for data_point in data_points:
            stored_data['Ticker'].append(release["contractId"])
            stored_data['ReleaseDate'].append(release["releaseDate"])

            stored_data['PeriodStart'].append(data_point["deliveryPeriod"]["startAt"])
            stored_data['PeriodEnd'].append(data_point["deliveryPeriod"]["endAt"])

            stored_data['USDperday'].append(data_point["derivedPrices"]["usdPerDay"]["spark"])
            stored_data['USDperdayMin'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMin"])
            stored_data['USDperdayMax'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMax"])
            
    historical_df = pd.DataFrame(stored_data)
    
    historical_df["USDperday"] = pd.to_numeric(historical_df["USDperday"])
    historical_df["USDperdayMax"] = pd.to_numeric(historical_df["USDperdayMax"])
    historical_df["USDperdayMin"] = pd.to_numeric(historical_df["USDperdayMin"])

    historical_df["ReleaseDate"] = pd.to_datetime(historical_df["ReleaseDate"])
    historical_df["PeriodStart"] = pd.to_datetime(historical_df["PeriodStart"])
    historical_df["PeriodEnd"] = pd.to_datetime(historical_df["PeriodEnd"])
    historical_df['CalMonth'] = historical_df['PeriodStart'].dt.strftime("%b-%Y")
        
    historical_df['TimeDiff'] = (historical_df['ReleaseDate'] - historical_df['PeriodStart']).dt.days

    return historical_df

### Call those functions for Spark30FFA and Spark25FFA

We call the function defined above and create two dataframes:

- spark25ffa - storing all historical Spark25FFA data
- spark30ffa - storing all historical Spark30FFA data

In [ ]:
spark25ffa = fetch_ffa_prices(tickers[4], 900)

spark30ffa = fetch_ffa_prices(tickers[8], 900)

# FFA Contract Evolution + Seasonality

Compare Spark30FFA December contracts from the past 3 years, and track how these contracts have priced in the year leading up to contract settlement

## Sorting Data

We then create new columns to categorise the data we have. By creating the 'Day of Year' column, we can plot the yearly data on the same x-axis range.

In [ ]:
def sort_years(df):
    
    reldates = df['ReleaseDate'].to_list()
    startdates = df['PeriodStart'].to_list()
    calmonths = df['CalMonth'].to_list()

    dayofyear = []
    
    for r in reldates:
        ir = reldates.index(r)
        if r.year - startdates[ir].year == -1:
            dayofyear.append(r.timetuple().tm_yday - 365)
        elif r.year - startdates[ir].year == -2:
            dayofyear.append(r.timetuple().tm_yday - 730)
        else:
            dayofyear.append(r.timetuple().tm_yday)
            
    df['DayofYear'] = dayofyear

    return df

spark30ffa = sort_years(spark30ffa)

## Group Dataframe by CalMonth and choose contracts to compare

In [ ]:
# Grouping the Dataframe by Contract
groups = spark30ffa.groupby('CalMonth')
years = list(spark30ffa['ReleaseDate'].dt.year.unique())

### Input which month you'd like to compare contracts for

In [ ]:
month = 'Dec'

# Plotting

In [ ]:
## Plotting
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style()
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(14, 7))

max_dates = []

# iterating through years and plotting the data
for y in years:
    ydf = groups.get_group(month + '-' + str(y))
    ydf = sort_years(ydf)
    ax.plot(ydf["DayofYear"], ydf["USDperday"], label=month + str(y))
    max_dates.append(ydf["DayofYear"].max())

plt.xlabel("Release Date")
plt.ylabel("Cost in USD/day")

# Setting custom x-axis ticks location and labels.
#xlabels = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December', 'Year End']
xlabels = ['Y-1', 'March', 'May', 'July', 'September', 'November', 'Y+0', 'March', 'May', 'July', 'September', 'November', 'Year End']

# xpos gives the first day of every other month in terms of 'day of year'
#xpos = [0,32,60,91,121,152,182,213,244,274,305,335, 365]
xpos = [-365,-305,-244,-183,-121,-60,0,60,121,182,244,305,365]

current_values = plt.gca().get_yticks()
plt.gca().set_yticklabels(['$ {:,.0f}'.format(x) for x in current_values])

plt.xticks(xpos, xlabels)

plt.xlim(-365,max(max_dates))

# Setting the graph legend
plt.legend()
